# 01 — Hello RAG: Your First End-to-End Pipeline

**By the end of this notebook, you can load files, split them into chunks, index vectors, retrieve top matches, and generate cited answers.**

### Learning Objectives
- Assemble the complete 5-stage RAG architecture (Load -> Split -> Embed & Store -> Retrieve -> Generate).
- Inspect real LangChain Document objects, chunk sizes, and vector similarity search results.
- Run live question answering against two real Novasphere documents.

### Prerequisites
- Notebook 00 (`00_why_rag.ipynb`)

## 1. Setup
Let's connect this notebook to our shared project helpers (`utils/models.py`), load `.env` (defaulting to free offline `local` mode), and initialize our LLM and embeddings.

In [1]:
# !pip install -q -r ../requirements.txt  # Uncomment if running in a fresh remote environment
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'utils').exists() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from utils.models import get_llm, get_embeddings, resolve_project_root
project_root = resolve_project_root()
data_dir = project_root / 'data'
llm = get_llm()
embeddings = get_embeddings()
print(f'Ready! Root={project_root.name} | LLM={type(llm).__name__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready! Root=rag-from-scratch-course | LLM=LocalGroundedChatModel


## 2. Concept: The 5-Stage RAG Pipeline

Every RAG system follows two phases: an **offline Indexing phase** (prepare your library shelf once) and a **real-time Query phase** (find answers on demand):

1. **Loader**: Reads files (`.txt`, `.md`, `.pdf`) into uniform **Documents** (text + metadata card).
2. **Text Splitter**: Slices long documents into short **Chunks** (~350 characters) so search hits pinpoint exact paragraphs.
3. **Embedding + Vector Store**: Turns each chunk into a numeric meaning coordinate (**Embedding**) inside an **InMemoryVectorStore** (a smart semantic filing cabinet).
4. **Retriever**: Embeds your user question and pulls the top-k closest matching chunks.
5. **Generator (Prompt + LLM)**: Hands the chunks to the LLM to synthesize a clear, cited answer.

```
INDEXING (Offline):  Files --> [Loader] --> [Splitter] --> [Embeddings] --> [Vector Store]
                                                                                  |
QUERYING (Online) :  User Question --> [Embed Question] --> [Similarity Search] --+
                                                                    |
                                          Final Answer <-- [LLM] <--+ (Top-k Chunks)
```

## 3. Step-by-Step Build

### Step 1: Load Real Documents
Let's use LangChain's `TextLoader` to load both `company_handbook.txt` and `product_faq.md` into standard `Document` objects.

In [2]:
from langchain_community.document_loaders import TextLoader
docs = []
for name in ["company_handbook.txt", "product_faq.md"]:
    loaded = TextLoader(str(data_dir / name), encoding="utf-8").load()
    for d in loaded: d.metadata["source"] = name
    docs.extend(loaded)
print(f"Loaded {len(docs)} documents cleanly.")
        

Loaded 2 documents cleanly.


C:\Users\Rahat\AppData\Local\Temp\ipykernel_9556\1270477611.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


> **What just happened?**
> Notice each document carries both `.page_content` (the raw words) and `.metadata` (where it came from).

### Step 2: Split Documents Into Searchable Chunks
A whole handbook is too broad for precise lookup, so let's chop it into overlapping 380-character pieces with `RecursiveCharacterTextSplitter`.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=380, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f"Split {len(docs)} files into {len(chunks)} searchable chunks.")
print("Sample chunk metadata:", chunks[0].metadata)
        

Split 2 files into 18 searchable chunks.
Sample chunk metadata: {'source': 'company_handbook.txt'}


> **What just happened?**
> Chunking keeps each piece small enough to match specific topics while carrying its source tag forward.

### Step 3: Embed Chunks & Store in Vector Store
Now let's pass all chunks through our local embedding model into `InMemoryVectorStore` so they are searchable by meaning.

In [4]:
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("Vector store ready! Retriever configured for top k=3 chunks.")
        

Vector store ready! Retriever configured for top k=3 chunks.


> **What just happened?**
> Behind the scenes, every chunk became a 384-dimensional vector coordinate stored in memory.

### Step 4: Retrieve Relevant Chunks for a Question
Let's invoke our `retriever` on a question about ROS2 support and inspect what chunks come back.

In [5]:
from utils.helpers import pretty_print_docs
question = "Which ROS2 distributions are natively supported by Novasphere?"
hits = retriever.invoke(question)
pretty_print_docs(hits, max_chars=140)
        

[1] (product_faq.md)
    --- ## Edge Compatibility & ROS2 Support ### Which ROS2 distributions are natively supported by Novasphere Edge Agent? Novasphere Edge Agent...
[2] (company_handbook.txt)
    1. COMPANY OVERVIEW & GLOBAL HUBS Novasphere Dynamics builds Novasphere OS, an autonomous fleet orchestration and real-time telemetry platfo...
[3] (company_handbook.txt)
    5. CUSTOMER SUPPORT TIERS & OPERATING HOURS Novasphere provides tiered technical support based on your active commercial subscription: - Sta...


> **What just happened?**
> Our semantic retriever pinpointed `product_faq.md` containing Humble Hawksbill, Iron Irwini, and Jazzy Jalisco!

### Step 5: Generate a Grounded Answer
Finally, let's combine the retrieved chunk texts into `<context>` tags and run our LLM chain to produce the answer.

> **Teaching Note (E2):** This pipeline is assembled from scratch step by step for pedagogical clarity; the encapsulated production equivalent is available in utils/models.py and pp/app.py (NovasphereRAGEngine).

In [6]:
from utils.helpers import print_qa_result
context_text = "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in hits)
prompt = f"<context>\n{context_text}\n</context>\nQuestion: {question}"
answer = llm.invoke(prompt).content
print_qa_result(question, answer, hits)
        

Question : Which ROS2 distributions are natively supported by Novasphere?
Answer   : Based on the provided context: ### Which ROS2 distributions are natively supported by Novasphere Edge Agent? Novasphere Edge Agent v4+ natively supports three long-term/active ROS2 distributions on Ubuntu 22.04 and 24.04 ARM64/x86_64 targets: **Humble Hawksbill** **Iron Irwini** **Jazzy Jalisco** ## Edge Compatibility & ROS2 Support ### Which ROS2 distributions are natively supported by Novasphere Edge Agent? Novasphere Edge Agent v4+ natively supports three long-term/active ROS2 distributions on Ubuntu 22.04 and 24.04 ARM64/x86_64 targets: **Humble Hawksbill** **Iron Irwini** **Jazzy Jalisco**
Sources  : company_handbook.txt, product_faq.md


> **What just happened?**
> Congratulations! You just built a working, cited RAG system from scratch in 5 crisp steps.

## 4. Try It Yourself
Experiment by editing the starter ideas below before peeking at the solution!

1. **Ask about travel meals** — *Hint:* Invoke the same pipeline for the daily meal reimbursement limit without prior VP approval.
2. **Adjust k to 1 vs 4** — *Hint:* See whether changing `search_kwargs={'k': 1}` still catches multi-part answers.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>

Run the cell below to verify your solution.
</details>

In [7]:
q_try = "What is the maximum daily meal expense reimbursement without VP approval?"
try_hits = vector_store.similarity_search(q_try, k=2)
ctx_try = "\n".join(d.page_content for d in try_hits)
print(llm.invoke(f"<context>\n{ctx_try}\n</context>\nQuestion: {q_try}").content)
        

Based on the provided context: Travel Meal Reimbursement: Employees traveling on company business are reimbursed up to $150 per day for meals and non-alcoholic beverages without requiring prior VP approval. Meal expenses exceeding $150 per day require written pre-approval from the VP of Finance.


## 5. Common Mistakes
- Forgetting to pass `encoding='utf-8'` to `TextLoader` on Windows (causes UnicodeDecodeError on special symbols or non-Latin text).
- Re-embedding the entire corpus on every single user query instead of indexing once and querying repeatedly.
- Setting `k=1` on complex questions — often misses crucial details in neighboring paragraphs.

## 6. Recap
- RAG runs in two distinct phases: Index offline, Retrieve + Generate online.
- LangChain unifies loaders, splitters, embeddings, and retrievers under clean `.invoke()` methods.
- Small, focused chunks + citations let beginners trace exactly why the model answered the way it did.

## 7. Next Notebook
Next up (`02_loading_documents.ipynb`): Master real-world file loading across TXT, Markdown, PDF, Web pages, Unicode scripts, and broken files!

## 8. Video Chapter Outline
1. **From Idea to Working Pipeline** — Overview of the 5-stage indexing + retrieval diagram.
2. **Loading & Chunking Data** — Hands-on `TextLoader` + `RecursiveCharacterTextSplitter`.
3. **Indexing Into InMemoryVectorStore** — How chunks become searchable vectors in 2 lines of Python.
4. **Retrieve + Generate Live** — Running our first real Q&A with cited sources.
5. **Recap & Homework** — Tuning k and testing your own questions.